In [3]:
# Run once in terminal/notebook if needed:
!pip install nltk spacy contractions scikit-learn gensim numpy transformers torch accelerate
!python -m spacy download en_core_web_sm

import re, nltk, spacy, contractions, torch
import numpy as np
import pandas as pd
from collections import Counter

from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.lm import Laplace
from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk import word_tokenize, pos_tag, CFG
from nltk.parse import ChartParser
from nltk.corpus import wordnet
from nltk.wsd import lesk

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from gensim.models import Word2Vec
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

nltk.download(["wordnet", "omw-1.4", "punkt", "punkt_tab", "averaged_perceptron_tagger_eng"], quiet=True)
nlp = spacy.load("en_core_web_sm")

# 1. Load dataset (Uncomment pd.read_csv in the exam if given a CSV file)
# df = pd.read_csv("dataset.csv")
# sentences = df["text"].dropna().tolist()

corpus = """
I don't like bugs http://test.com :( I like natural language processing :)
I like machine learning and language models.
The boy saw the boy with the telescope.
The food is delicious. The phone has a great screen.
Mr. Rahul bought a laptop. He loved the laptop. Mrs. Priya said she liked it.
"""
sentences = [s.strip() for s in corpus.strip().split("\n") if s.strip()]
print("Loaded Sentences:", len(sentences))

Defaulting to user installation because normal site-packages is not writeable

   -------- ------------------------------- 1/5 [anyascii]
   -------- ------------------------------- 1/5 [anyascii]
   -------- ------------------------------- 1/5 [anyascii]
   -------- ------------------------------- 1/5 [anyascii]
   -------- ------------------------------- 1/5 [anyascii]
   -------- ------------------------------- 1/5 [anyascii]
   ---------------- ----------------------- 2/5 [textsearch]
   -------------------------------- ------- 4/5 [accelerate]
   -------------------------------- ------- 4/5 [accelerate]
   -------------------------------- ------- 4/5 [accelerate]
   -------------------------------- ------- 4/5 [accelerate]
   -------------------------------- ------- 4/5 [accelerate]
   -------------------------------- ------- 4/5 [accelerate]
   -------------------------------- ------- 4/5 [accelerate]
   -------------------------------- ------- 4/5 [accelerate]
   ---------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


Defaulting to user installation because normal site-packages is not writeable
     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
      --------------------------------------- 0.3/12.8 MB ? eta -:--:--
     ---- ----------------------------------- 1.3/12.8 MB 6.4 MB/s eta 0:00:02
     ----- ---------------------------------- 1.8/12.8 MB 5.0 MB/s eta 0:00:03
     ------- -------------------------------- 2.4/12.8 MB 3.4 MB/s eta 0:00:04
     -------- ------------------------------- 2.6/12.8 MB 2.8 MB/s eta 0:00:04
     --------- ------------------------------ 2.9/12.8 MB 2.5 MB/s eta 0:00:04
     ---------- ----------------------------- 3.4/12.8 MB 2.4 MB/s eta 0:00:04
     ----------- ---------------------------- 3.7/12.8 MB 2.3 MB/s eta 0:00:04
     ------------ --------------------------- 3.9/12.8 MB 2.1 MB/s eta 0:00:05
     ------------ --------------------------- 3.9/12.8 MB 2.1 MB/s eta 0:00:05
     ------------- -------------------------- 4.2/12.8 MB 1.9 MB/s 

C:\Users\karth\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded Sentences: 5


In [4]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r"https?://\S+|www\.\S+", "", text)          # Remove URLs
    text = contractions.fix(text)                              # Expand contractions
    text = text.replace(":)", " <happy> ").replace(":(", " <sad> ")
    return re.findall(r"<happy>|<sad>|[a-z]+", text)           # Regex tokenise

tokenized_sentences = [preprocess(s) for s in sentences]
print("Tokenized:", tokenized_sentences[0])

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

all_words = [w for s in tokenized_sentences for w in s]
print("Stemmed   :", [stemmer.stem(w) for w in all_words[:10]])
print("Lemmatized:", [lemmatizer.lemmatize(w, pos="v") for w in all_words[:10]])

Tokenized: ['i', 'do', 'not', 'like', 'bugs', '<sad>', 'i', 'like', 'natural', 'language', 'processing', '<happy>']
Stemmed   : ['i', 'do', 'not', 'like', 'bug', '<sad>', 'i', 'like', 'natur', 'languag']
Lemmatized: ['i', 'do', 'not', 'like', 'bug', '<sad>', 'i', 'like', 'natural', 'language']


In [5]:
train_data, vocab = padded_everygram_pipeline(2, tokenized_sentences)
lm = Laplace(2)
lm.fit(train_data, vocab)

# 1. Bigram probabilities: P(w2 | w1) = (C + 1) / (C(prev) + V)
print("P(like | i) =", round(lm.score("like", ["i"]), 4))

# 2. Sentence Probability
test_tokens = ["<s>"] + preprocess(sentences[1]) + ["</s>"]
sent_prob = 1.0
for i in range(1, len(test_tokens)):
    sent_prob *= lm.score(test_tokens[i], [test_tokens[i - 1]])
print("Sentence Probability:", sent_prob)

# 3. Perplexity (inbuilt lm.perplexity)
test_bigrams = list(nltk.bigrams(test_tokens))
print("Perplexity:", round(lm.perplexity(test_bigrams), 4))

# 4. Generate/Sample sentence with fixed seed (1-line inbuilt call)
generated = lm.generate(8, text_seed=["<s>"], random_seed=42)
print("Generated Sentence:", " ".join(w for w in generated if w != "</s>"))

P(like | i) = 0.0667
Sentence Probability: 4.023213678145146e-11
Perplexity: 19.9264
Generated Sentence: the boy saw the phone has a great


In [6]:
# 1. POS Tagging: Open-class vs Closed-class
words = word_tokenize(sentences[1])
tagged = pos_tag(words)

open_class = [(w, t) for w, t in tagged if t.startswith(("NN", "VB", "JJ", "RB"))]
closed_class = [(w, t) for w, t in tagged if t.startswith(("DT", "IN", "PRP", "CC"))]

print("POS Tags:", tagged)
print("Open-class:", open_class)
print("Closed-class:", closed_class)

# 2. CFG + ChartParser for "...with the telescope" (2 trees for PP-attachment ambiguity)
grammar = CFG.fromstring("""
S -> NP VP
NP -> Det N | Det N PP
VP -> V NP | VP PP
PP -> P NP
Det -> 'the'
N -> 'boy' | 'telescope'
V -> 'saw'
P -> 'with'
""")

parser = ChartParser(grammar)
cfg_tokens = "the boy saw the boy with the telescope".split()

for i, tree in enumerate(parser.parse(cfg_tokens), 1):
    print(f"--- Parse Tree {i} ---")
    tree.pretty_print()

POS Tags: [('I', 'PRP'), ('like', 'VBP'), ('machine', 'NN'), ('learning', 'NN'), ('and', 'CC'), ('language', 'NN'), ('models', 'NNS'), ('.', '.')]
Open-class: [('like', 'VBP'), ('machine', 'NN'), ('learning', 'NN'), ('language', 'NN'), ('models', 'NNS')]
Closed-class: [('I', 'PRP'), ('and', 'CC')]
--- Parse Tree 1 ---
                 S                                
      ___________|_______                          
     |                   VP                       
     |            _______|________                 
     |           VP               PP              
     |        ___|___         ____|___             
     NP      |       NP      |        NP          
  ___|___    |    ___|___    |     ___|______      
Det      N   V  Det      N   P   Det         N    
 |       |   |   |       |   |    |          |     
the     boy saw the     boy with the     telescope

--- Parse Tree 2 ---
                 S                            
      ___________|___                        

In [7]:
for s in sentences[3:5]:
    doc = nlp(s)
    print("\nSentence:", s)
    for t in doc:
        print(f"{t.text:10} POS: {t.pos_:6} Dep: {t.dep_:8} Head: {t.head.text}")

    pairs = []
    for t in doc:
        # Rule 1: Adjective modifier (amod), e.g., "great screen"
        if t.dep_ == "amod":
            pairs.append((t.head.text, t.text))

        # Rule 2: Adjectival complement (acomp) with nsubj, e.g., "food is delicious"
        if t.dep_ == "acomp":
            for child in t.head.children:
                if child.dep_ == "nsubj":
                    pairs.append((child.text, t.text))

    print("Aspect-Opinion Pairs:", pairs)


Sentence: The food is delicious. The phone has a great screen.
The        POS: DET    Dep: det      Head: food
food       POS: NOUN   Dep: nsubj    Head: is
is         POS: AUX    Dep: ROOT     Head: is
delicious  POS: ADJ    Dep: acomp    Head: is
.          POS: PUNCT  Dep: punct    Head: is
The        POS: DET    Dep: det      Head: phone
phone      POS: NOUN   Dep: nsubj    Head: has
has        POS: VERB   Dep: ROOT     Head: has
a          POS: DET    Dep: det      Head: screen
great      POS: ADJ    Dep: amod     Head: screen
screen     POS: NOUN   Dep: dobj     Head: has
.          POS: PUNCT  Dep: punct    Head: has
Aspect-Opinion Pairs: [('food', 'delicious'), ('screen', 'great')]

Sentence: Mr. Rahul bought a laptop. He loved the laptop. Mrs. Priya said she liked it.
Mr.        POS: PROPN  Dep: compound Head: Rahul
Rahul      POS: PROPN  Dep: nsubj    Head: bought
bought     POS: VERB   Dep: ROOT     Head: bought
a          POS: DET    Dep: det      Head: laptop
laptop     P

In [8]:
# 1. WordNet Synsets
for syn in wordnet.synsets("language")[:3]:
    print(syn.name(), ":", syn.definition())

# 2. Wu-Palmer Similarity
s1 = wordnet.synsets("machine")[0]
s2 = wordnet.synsets("computer")[0]
print("\nWu-Palmer Similarity:", round(s1.wup_similarity(s2), 3))

# 3. Lesk WSD
sense = lesk(word_tokenize("I like machine learning"), "learning")
print("Lesk Sense:", sense.name(), "-", sense.definition())

# 4. TF-IDF + Cosine Similarity + Top-k Retrieval
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(sentences)
query_vec = vectorizer.transform(["machine learning"])

sims = cosine_similarity(query_vec, tfidf_matrix)[0]
top_k = sims.argsort()[::-1][:3]

print("\nTop-3 TF-IDF Matches:")
for i in top_k:
    print(f"Score: {sims[i]:.3f} | {sentences[i]}")

language.n.01 : a systematic means of communicating by the use of sounds or conventional symbols
speech.n.02 : (language) communication by word of mouth
lyric.n.01 : the text of a popular song or musical-comedy number

Wu-Palmer Similarity: 0.941
Lesk Sense: learning.n.01 - the cognitive process of acquiring skill or knowledge

Top-3 TF-IDF Matches:
Score: 0.614 | I like machine learning and language models.
Score: 0.000 | Mr. Rahul bought a laptop. He loved the laptop. Mrs. Priya said she liked it.
Score: 0.000 | The food is delicious. The phone has a great screen.


In [9]:
# 1. PPMI with Window +/- 2
words = [w for s in tokenized_sentences for w in s]
pairs = Counter()

for i, w in enumerate(words):
    for j in range(max(0, i - 2), min(len(words), i + 3)):
        if i != j:
            pairs[(w, words[j])] += 1

total = sum(pairs.values())
w_counts = Counter(words)

print("Sample PPMI Scores:")
for (w1, w2), c in list(pairs.items())[:6]:
    pmi = np.log2((c / total) / ((w_counts[w1] / len(words)) * (w_counts[w2] / len(words))))
    print(f"({w1}, {w2}) -> PPMI: {max(0, round(pmi, 3))}")

# 2. Word2Vec Skip-Gram (sg=1)
w2v = Word2Vec(sentences=tokenized_sentences, vector_size=50, window=2,
               min_count=1, sg=1, seed=42, workers=1, epochs=50)
print("\nMost similar to 'learning':", w2v.wv.most_similar("learning", topn=3))

Sample PPMI Scores:
(i, do) -> PPMI: 2.158
(i, not) -> PPMI: 2.158
(do, i) -> PPMI: 2.158
(do, not) -> PPMI: 3.743
(do, like) -> PPMI: 2.158
(not, i) -> PPMI: 2.158

Most similar to 'learning': [('and', 0.33537665009498596), ('liked', 0.3066946864128113), ('<sad>', 0.22451967000961304)]


In [10]:
doc = nlp(sentences[-1])  # "Mr. Rahul bought a laptop. He loved the laptop. Mrs. Priya said she liked it."

print("Named Entities:", [(e.text, e.label_) for e in doc.ents])
print("Noun Chunks   :", [c.text for c in doc.noun_chunks])
print("Pronouns      :", [t.text for t in doc if t.pos_ == "PRON"])

male_words = {"mr.", "mr", "rahul", "john", "he", "him", "his", "boy"}
female_words = {"mrs.", "mrs", "ms.", "priya", "sara", "she", "her", "girl"}
plural_words = {"they", "them", "their"}

def get_info(span):
    words = set(span.text.lower().split())
    if words & male_words:
        gender = "M"
    elif words & female_words:
        gender = "F"
    else:
        gender = ""
    number = "P" if (words & plural_words or span.root.tag_ in ("NNS", "NNPS")) else "S"
    return {
        "text": span.text,
        "head": span.root.text.lower(),
        "gender": gender,
        "number": number,
        "sent": list(doc.sents).index(span.sent)
    }

# Extract all noun_chunks as mentions (includes names, nouns, and pronouns in spaCy)
mentions = [get_info(c) for c in doc.noun_chunks]

def score_pair(m1, m2):
    score = 0
    if m1["gender"] and m2["gender"]:
        score += 2 if m1["gender"] == m2["gender"] else -2
    score += 2 if m1["number"] == m2["number"] else -2
    score += max(0, 3 - abs(m1["sent"] - m2["sent"]))  # Sentence distance
    if m1["head"] == m2["head"]:
        score += 2                                     # Head-word match
    return score

print("\nMention-Pair Scores:")
for i in range(len(mentions)):
    for j in range(i + 1, len(mentions)):
        print(f"{mentions[i]['text']} <-> {mentions[j]['text']} : {score_pair(mentions[i], mentions[j])}")

# Build Coreference Chains
chains = []
for m in mentions:
    best_chain, best_score = None, -999
    for c in chains:
        s = score_pair(c[-1], m)
        if s > best_score:
            best_score, best_chain = s, c
    if best_chain and best_score >= 4:
        best_chain.append(m)
    else:
        chains.append([m])

print("\nCoreference Chains:")
for i, c in enumerate(chains, 1):
    print(f"Chain {i}: {[m['text'] for m in c]}")

Named Entities: [('Rahul', 'PERSON'), ('Priya', 'PERSON')]
Noun Chunks   : ['Mr. Rahul', 'a laptop', 'He', 'the laptop', 'Mrs. Priya', 'she', 'it']
Pronouns      : ['He', 'she', 'it']

Mention-Pair Scores:
Mr. Rahul <-> a laptop : 5
Mr. Rahul <-> He : 6
Mr. Rahul <-> the laptop : 4
Mr. Rahul <-> Mrs. Priya : 1
Mr. Rahul <-> she : 1
Mr. Rahul <-> it : 3
a laptop <-> He : 4
a laptop <-> the laptop : 6
a laptop <-> Mrs. Priya : 3
a laptop <-> she : 3
a laptop <-> it : 3
He <-> the laptop : 5
He <-> Mrs. Priya : 2
He <-> she : 2
He <-> it : 4
the laptop <-> Mrs. Priya : 4
the laptop <-> she : 4
the laptop <-> it : 4
Mrs. Priya <-> she : 7
Mrs. Priya <-> it : 5
she <-> it : 5

Coreference Chains:
Chain 1: ['Mr. Rahul', 'a laptop', 'He', 'the laptop', 'Mrs. Priya', 'she', 'it']


In [11]:
fill_mask = pipeline("fill-mask", model="bert-base-uncased")
print("Fill-Mask Top Result:", fill_mask("I like natural language [MASK].")[0]["sequence"])

sentiment = pipeline("sentiment-analysis")
print("Sentiment (+):", sentiment("The food is delicious."))
print("Sentiment (-):", sentiment("The service is terrible."))

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 264.06it/s]
[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Fill-Mask Top Result: i like natural language speakers.


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 313.76it/s]


Sentiment (+): [{'label': 'POSITIVE', 'score': 0.9998831748962402}]
Sentiment (-): [{'label': 'NEGATIVE', 'score': 0.9996910095214844}]


In [12]:
texts_9 = [
    "The food is delicious", "I love this movie", "The service is excellent",
    "This product is amazing", "I enjoyed the experience",
    "The food is terrible", "I hate this movie", "The service is poor", "This product is bad"
]
labels_9 = [1, 1, 1, 1, 1, 0, 0, 0, 0]

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

X_tr, X_te, y_tr, y_te = train_test_split(texts_9, labels_9, test_size=0.33, random_state=42, stratify=labels_9)

def make_dataset(txts, lbls):
    enc = tokenizer(txts, truncation=True, padding=True)
    return [{"input_ids": enc["input_ids"][i], "attention_mask": enc["attention_mask"][i], "labels": lbls[i]} for i in range(len(lbls))]

def compute_metrics(eval_pred):
    preds = eval_pred.predictions.argmax(axis=-1)
    return {"accuracy": accuracy_score(eval_pred.label_ids, preds), "f1": f1_score(eval_pred.label_ids, preds, zero_division=0)}

args = TrainingArguments(output_dir="./out", num_train_epochs=3, per_device_train_batch_size=2, report_to="none", save_strategy="no")
trainer = Trainer(model=model, args=args, train_dataset=make_dataset(X_tr, y_tr), eval_dataset=make_dataset(X_te, y_te), compute_metrics=compute_metrics)

trainer.train()
res = trainer.evaluate()
print("Accuracy:", res["eval_accuracy"], "| F1:", res["eval_f1"])

# Predict on new text
new_in = tokenizer(["The food is wonderful"], return_tensors="pt").to(model.device)
pred = model(**new_in).logits.argmax(dim=-1).item()
print("New Prediction:", "Positive" if pred == 1 else "Negative")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 269.22it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoin

Step,Training Loss


Training Loss,Validation Loss,Step,Accuracy,F1
No log,0.731736,9,0.333333,0.000000


Accuracy: 0.3333333333333333 | F1: 0.0
New Prediction: Negative


In [71]:
import nltk
from nltk import CFG, ChartParser

nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)

# Step 1: Enter a sentence
sentence = input("Enter sentence: ").lower()
tokens = nltk.word_tokenize(sentence)

# Step 2: Identify parts of speech
tagged = nltk.pos_tag(tokens)
print("POS Tags:", tagged)

# Step 3: Generate word rules automatically
rules = {"Det": set(), "N": set(), "V": set(), "P": set()}

for word, tag in tagged:
    if tag == "DT":
        rules["Det"].add(word)
    elif tag.startswith("NN"):
        rules["N"].add(word)
    elif tag.startswith("VB"):
        rules["V"].add(word)
    elif tag == "IN":
        rules["P"].add(word)

# Step 4: Generate CFG grammar
grammar_text = """
S -> NP VP
NP -> Det N | NP PP
VP -> V NP | VP PP
PP -> P NP
"""

for category, words in rules.items():
    if words:
        grammar_text += (
            category + " -> " +
            " | ".join(repr(w) for w in sorted(words)) + "\n"
        )

print("\nGenerated CFG Grammar:")
print(grammar_text)

# Step 5: Generate parse trees
grammar = CFG.fromstring(grammar_text)
parser = ChartParser(grammar)
trees = list(parser.parse(tokens))

print("Number of parse trees:", len(trees))

for tree in trees:
    tree.pretty_print()

Enter sentence:  The boy saw the telescope with the telescope


POS Tags: [('the', 'DT'), ('boy', 'NN'), ('saw', 'VBD'), ('the', 'DT'), ('telescope', 'NN'), ('with', 'IN'), ('the', 'DT'), ('telescope', 'NN')]

Generated CFG Grammar:

S -> NP VP
NP -> Det N | NP PP
VP -> V NP | VP PP
PP -> P NP
Det -> 'the'
N -> 'boy' | 'telescope'
V -> 'saw'
P -> 'with'

Number of parse trees: 2
                 S                                      
      ___________|__________                             
     |                      VP                          
     |            __________|___________                 
     |           VP                     PP              
     |        ___|___               ____|___             
     NP      |       NP            |        NP          
  ___|___    |    ___|______       |     ___|______      
Det      N   V  Det         N      P   Det         N    
 |       |   |   |          |      |    |          |     
the     boy saw the     telescope with the     telescope

                 S                               

In [14]:
import re, math, random, nltk, contractions, torch
import numpy as np
import pandas as pd
from collections import Counter

from nltk.lm import Laplace
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import wordnet, word_tokenize, pos_tag, CFG
from nltk.parse import ChartParser
from nltk.corpus import wordnet
from nltk.wsd import lesk
from nltk.lm.preprocessing import padded_everygram_pipeline 

import re, math, random, contractions, torch
from collections import Counter

import nltk
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.lm import Laplace
from nltk.lm.preprocessing import padded_everygram_pipeline
from nltk.wsd import lesk
from nltk import word_tokenize,pos_tag,CFG
from nltk.parse import ChartParser
from nltk.corpus import wordnet

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from gensim.models import Word2Vec
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments


In [15]:
nltk.download(['wordnet','omw-1.4','punkt','punkt_tab','averaged_perceptron_tagger_eng'])
nlp=spacy.load('en_core_web_sm')

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\karth\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\karth\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\karth\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\karth\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\karth\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [16]:
corpus = """
I don't like bugs http://test.com :( I like natural language processing :)
I like machine learning and language models.
The boy saw the boy with the telescope.
The food is delicious. The phone has a great screen.
Mr. Rahul bought a laptop. He loved the laptop. Mrs. Priya said she liked it.
"""

In [17]:
sentences=[s.strip() for s in corpus.strip().split("\n") if s.strip()]
print(sentences)

["I don't like bugs http://test.com :( I like natural language processing :)", 'I like machine learning and language models.', 'The boy saw the boy with the telescope.', 'The food is delicious. The phone has a great screen.', 'Mr. Rahul bought a laptop. He loved the laptop. Mrs. Priya said she liked it.']


In [18]:
def preprocess(text):
    text=text.lower()
    text=re.sub(r"http\S+|www\.\S+","",text)
    text = contractions.fix(text)
    text = text.replace(":)", " <happy> ").replace(":(", " <sad> ")
    return re.findall(r"<happy>|<sad>|[a-z]+", text) 

In [19]:
tokenized_sentences = [preprocess(s) for s in sentences]
print("Tokenized:", tokenized_sentences[0])

Tokenized: ['i', 'do', 'not', 'like', 'bugs', '<sad>', 'i', 'like', 'natural', 'language', 'processing', '<happy>']


In [20]:
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

all_words = [w for s in tokenized_sentences for w in s]
print("Stemmed   :", [stemmer.stem(w) for w in all_words[:10]])
print("Lemmatized:", [lemmatizer.lemmatize(w, pos="v") for w in all_words[:10]])

Stemmed   : ['i', 'do', 'not', 'like', 'bug', '<sad>', 'i', 'like', 'natur', 'languag']
Lemmatized: ['i', 'do', 'not', 'like', 'bug', '<sad>', 'i', 'like', 'natural', 'language']


In [21]:
sentences=[s.strip() for s in corpus.strip().split('\n') if s.strip()]
sentences

["I don't like bugs http://test.com :( I like natural language processing :)",
 'I like machine learning and language models.',
 'The boy saw the boy with the telescope.',
 'The food is delicious. The phone has a great screen.',
 'Mr. Rahul bought a laptop. He loved the laptop. Mrs. Priya said she liked it.']

In [22]:
nlp=spacy.load('en_core_web_sm')

In [24]:
def preprocess(text):
    text=text.lower()
    text=re.sub('http\S+|www\.\S+','',text)
    text=contractions.fix(text)
    text=text.replace(':)',' <happy> ').replace(':(',' <sad> ')
    return re.findall(r'<happy>|<sad>|[a-z]+',text)
tokenized_sentences=[preprocess(s) for s in sentences]
print("Tokenized:", tokenized_sentences)

Tokenized: [['i', 'do', 'not', 'like', 'bugs', '<sad>', 'i', 'like', 'natural', 'language', 'processing', '<happy>'], ['i', 'like', 'machine', 'learning', 'and', 'language', 'models'], ['the', 'boy', 'saw', 'the', 'boy', 'with', 'the', 'telescope'], ['the', 'food', 'is', 'delicious', 'the', 'phone', 'has', 'a', 'great', 'screen'], ['mr', 'rahul', 'bought', 'a', 'laptop', 'he', 'loved', 'the', 'laptop', 'mrs', 'priya', 'said', 'she', 'liked', 'it']]


<>:3: SyntaxWarning: invalid escape sequence '\S'
<>:3: SyntaxWarning: invalid escape sequence '\S'
C:\Users\karth\AppData\Local\Temp\ipykernel_2940\4113190230.py:3: SyntaxWarning: invalid escape sequence '\S'
  text=re.sub('http\S+|www\.\S+','',text)


In [25]:
stemmer=PorterStemmer()
lemmatizer=WordNetLemmatizer()
all_words = [w for s in tokenized_sentences for w in s]
print([stemmer.stem(w) for w in all_words])
print([lemmatizer.lemmatize(w,pos='v') for w in all_words])

['i', 'do', 'not', 'like', 'bug', '<sad>', 'i', 'like', 'natur', 'languag', 'process', '<happy>', 'i', 'like', 'machin', 'learn', 'and', 'languag', 'model', 'the', 'boy', 'saw', 'the', 'boy', 'with', 'the', 'telescop', 'the', 'food', 'is', 'delici', 'the', 'phone', 'ha', 'a', 'great', 'screen', 'mr', 'rahul', 'bought', 'a', 'laptop', 'he', 'love', 'the', 'laptop', 'mr', 'priya', 'said', 'she', 'like', 'it']
['i', 'do', 'not', 'like', 'bug', '<sad>', 'i', 'like', 'natural', 'language', 'process', '<happy>', 'i', 'like', 'machine', 'learn', 'and', 'language', 'model', 'the', 'boy', 'saw', 'the', 'boy', 'with', 'the', 'telescope', 'the', 'food', 'be', 'delicious', 'the', 'phone', 'have', 'a', 'great', 'screen', 'mr', 'rahul', 'buy', 'a', 'laptop', 'he', 'love', 'the', 'laptop', 'mrs', 'priya', 'say', 'she', 'like', 'it']


In [26]:
train_data,vocab=padded_everygram_pipeline(2,tokenized_sentences)
print(train_data,vocab)

<generator object padded_everygram_pipeline.<locals>.<genexpr> at 0x00000209C8EBBA40> <itertools.chain object at 0x00000209BF397100>


In [27]:
lm=Laplace(2)
lm.fit(train_data,vocab)

In [28]:
print("P(like|i)=",round(lm.score("like",['i']),4))

P(like|i)= 0.0667


In [33]:
test_tokens=['<s>']+preprocess(sentences[1])+['</s>']
sent_prob=1.0
for i in range(1,len(test_tokens)):
    sent_prob*=lm.score(test_tokens[i],[test_tokens[i-1]])
print(round(sent_prob,20))

4.023213678e-11


In [34]:
test_bigrams=list(nltk.bigrams(test_tokens))
print(round(lm.perplexity(test_bigrams),4))

19.9264


In [37]:
gen=lm.generate(10,text_seed=['<s>'],random_seed=42)
print(gen)

['the', 'boy', 'saw', 'the', 'phone', 'has', 'a', 'great', 'screen', '</s>']


In [41]:
def preprocess(text):
    text=text.lower()
    text=re.sub(r'http\S+|www\.\S+','',text)
    text=contractions.fix(text)
    text=text.replace(':)','<happy>').replace(':(','<sad>')
    return re.findall(r'<happy>|<sad>|[a-z]+',text)
tokenized_sent=[preprocess(s) for s in sentences]
print(tokenized_sent)

[['i', 'do', 'not', 'like', 'bugs', '<sad>', 'i', 'like', 'natural', 'language', 'processing', '<happy>'], ['i', 'like', 'machine', 'learning', 'and', 'language', 'models'], ['the', 'boy', 'saw', 'the', 'boy', 'with', 'the', 'telescope'], ['the', 'food', 'is', 'delicious', 'the', 'phone', 'has', 'a', 'great', 'screen'], ['mr', 'rahul', 'bought', 'a', 'laptop', 'he', 'loved', 'the', 'laptop', 'mrs', 'priya', 'said', 'she', 'liked', 'it']]


In [42]:
stemmer=PorterStemmer()
lemmatizer=WordNetLemmatizer()
all_words=[w for s in tokenized_sent for w in s]
print([stemmer.stem(w) for w in all_words])
print([lemmatizer.lemmatize(w,pos='v') for w in all_words])

['i', 'do', 'not', 'like', 'bug', '<sad>', 'i', 'like', 'natur', 'languag', 'process', '<happy>', 'i', 'like', 'machin', 'learn', 'and', 'languag', 'model', 'the', 'boy', 'saw', 'the', 'boy', 'with', 'the', 'telescop', 'the', 'food', 'is', 'delici', 'the', 'phone', 'ha', 'a', 'great', 'screen', 'mr', 'rahul', 'bought', 'a', 'laptop', 'he', 'love', 'the', 'laptop', 'mr', 'priya', 'said', 'she', 'like', 'it']
['i', 'do', 'not', 'like', 'bug', '<sad>', 'i', 'like', 'natural', 'language', 'process', '<happy>', 'i', 'like', 'machine', 'learn', 'and', 'language', 'model', 'the', 'boy', 'saw', 'the', 'boy', 'with', 'the', 'telescope', 'the', 'food', 'be', 'delicious', 'the', 'phone', 'have', 'a', 'great', 'screen', 'mr', 'rahul', 'buy', 'a', 'laptop', 'he', 'love', 'the', 'laptop', 'mrs', 'priya', 'say', 'she', 'like', 'it']


In [43]:
train_data,vocab=padded_everygram_pipeline(2,tokenized_sent)
lm=Laplace(2)
lm.fit(train_data,vocab)
print(lm)

In [44]:
print(lm.score('like',['i']))

0.06666666666666667


In [45]:
test_tokens=['<s>']+preprocess(sentences[1])+['</s>']
print(test_tokens)

['<s>', 'i', 'like', 'machine', 'learning', 'and', 'language', 'models', '</s>']


In [46]:
sent_prob=1.0
for i in range(1,len(test_tokens)):
    sent_prob*=lm.score(test_tokens[i],[test_tokens[i-1]])
print(sent_prob)

4.023213678145146e-11


In [47]:
test_bigrams=list(nltk.bigrams(test_tokens))
print(lm.perplexity(test_bigrams))

19.92637791079917


In [55]:
words=word_tokenize(sentences[1])
tagged=pos_tag(words)
open_class=[(w,t) for w,t in tagged if t.startswith(('NN','VB','JJ','RB'))]
closed_class=[(w,t) for w,t in tagged if t.startswith(('DT','IN','PRP','CC'))]
grammar=CFG.fromstring("""
S -> NP VP
NP -> Det N | Det N PP
VP -> V NP | VP PP
PP -> P NP 
Det -> 'the'
N -> 'boy' | 'telescope'
V -> 'saw'
P -> 'with'
""")
parser=nltk.ChartParser(grammar)
cfg_tokens="the boy saw the boy with the telescope".split()
print('hello')
for i,tree in enumerate(parser.parse(cfg_tokens),1):
    print(f"--- Parse Tree {i} ---")
    tree.pretty_print()
    

hello
--- Parse Tree 1 ---
                 S                                
      ___________|_______                          
     |                   VP                       
     |            _______|________                 
     |           VP               PP              
     |        ___|___         ____|___             
     NP      |       NP      |        NP          
  ___|___    |    ___|___    |     ___|______      
Det      N   V  Det      N   P   Det         N    
 |       |   |   |       |   |    |          |     
the     boy saw the     boy with the     telescope

--- Parse Tree 2 ---
                 S                            
      ___________|___                          
     |               VP                       
     |        _______|___                      
     |       |           NP                   
     |       |    _______|____                 
     |       |   |   |        PP              
     |       |   |   |    ____|___             
     

In [70]:
for s in sentences[3:5]:
    doc=nlp(s)
    print(s)
    for t in doc:
        print(f"{t.text:10} pos:{t.pos_:10} dep:{t.dep_:10} head:{t.head.text}")
    pairs=[]
    for t in doc:
        if t.dep_=="amod":
            pairs.append((t.head.text,t.text))
        if t.dep_=="acomp":
            for child in t.head.children:
                if child.dep_=="nsubj":
                    pairs.append((child.text,t.text))
    print("Aspect-Opinion Pairs:", pairs)

The food is delicious. The phone has a great screen.
The        pos:DET        dep:det        head:food
food       pos:NOUN       dep:nsubj      head:is
is         pos:AUX        dep:ROOT       head:is
delicious  pos:ADJ        dep:acomp      head:is
.          pos:PUNCT      dep:punct      head:is
The        pos:DET        dep:det        head:phone
phone      pos:NOUN       dep:nsubj      head:has
has        pos:VERB       dep:ROOT       head:has
a          pos:DET        dep:det        head:screen
great      pos:ADJ        dep:amod       head:screen
screen     pos:NOUN       dep:dobj       head:has
.          pos:PUNCT      dep:punct      head:has
Aspect-Opinion Pairs: [('food', 'delicious'), ('screen', 'great')]
Mr. Rahul bought a laptop. He loved the laptop. Mrs. Priya said she liked it.
Mr.        pos:PROPN      dep:compound   head:Rahul
Rahul      pos:PROPN      dep:nsubj      head:bought
bought     pos:VERB       dep:ROOT       head:bought
a          pos:DET        dep:det      

In [69]:
grammar=CFG.fromstring("""
S -> NP VP
NP -> C NN | PRON
VP -> V S | V NP
C -> 'mrs.'
NN -> 'priya' 
PRON -> 'it' | 'she'
V -> 'said' | 'liked'
""")
parser=ChartParser(grammar)
cfg_tokens="mrs. priya said she liked it".split()
for i,tree in enumerate(parser.parse(cfg_tokens),1):
    print(tree)

(S
  (NP (C mrs.) (NN priya))
  (VP (V said) (S (NP (PRON she)) (VP (V liked) (NP (PRON it))))))


In [73]:
for syn in wordnet.synsets("language")[:4]:
    print(syn.name(),":",syn.definition())
    

language.n.01 : a systematic means of communicating by the use of sounds or conventional symbols
speech.n.02 : (language) communication by word of mouth
lyric.n.01 : the text of a popular song or musical-comedy number
linguistic_process.n.02 : the cognitive processes involved in producing and understanding linguistic communication


In [74]:
s1=wordnet.synsets("machine")[0]
s2=wordnet.synsets("computer")[0]
round(s1.wup_similarity(s2),3)

0.941

In [75]:
sense=lesk(word_tokenize("I like machine learning"),"learning")
print(sense.name(),sense.definition())

learning.n.01 the cognitive process of acquiring skill or knowledge


In [84]:
vectorizer=TfidfVectorizer()
tfidf_matrix=vectorizer.fit_transform(sentences)
query_vec=vectorizer.transform(["is"])
sims=cosine_similarity(query_vec,tfidf_matrix)[0]
top_k=sims.argsort()[::-1][:3]
for i in top_k:
    print(f"{sims[i]} | {sentences[i]}")

0.33721385976751955 | The food is delicious. The phone has a great screen.
0.0 | Mr. Rahul bought a laptop. He loved the laptop. Mrs. Priya said she liked it.
0.0 | The boy saw the boy with the telescope.


In [ ]:
words=[w for s in tokenized_sentences for w in s]
pairs=Counter()
for i,w in enumerate(words):
    for j in range(max(0,i-2),min(len(words),i+3)):
        for i!=j:
        pairs[(w,words[j])]+=1
total=sum(pairs.values())
w_counts=Counter(words)


In [85]:
train_data,vocab=padded_everygram_pipeline(2,tokenized_sentences)
lm=Laplace(2)
lm.fit(train_data,vocab)
print(lm.score("like","i"))

0.023809523809523808


In [87]:
test_tokens=['<s>']+preprocess(sentences[1])+['</s>']
prob=1.0
for i in range(1,len(test_tokens)):
    prob*=lm.score(test_tokens[i],[test_tokens[i-1]])
print(prob)

4.023213678145146e-11


In [88]:
lm.perplexity(test_tokens)

39.8205742818275

In [89]:
test_bigrams=list(nltk.bigrams(test_tokens))
lm.perplexity(test_bigrams)

19.92637791079917

In [90]:
gen=lm.generate(8,text_seed=['<s>'],random_seed=42)

In [91]:
gen

['the', 'boy', 'saw', 'the', 'phone', 'has', 'a', 'great']

In [93]:
for syn in wordnet.synsets("language")[:3]:
    print(syn.name(),":",syn.definition())

language.n.01 : a systematic means of communicating by the use of sounds or conventional symbols
speech.n.02 : (language) communication by word of mouth
lyric.n.01 : the text of a popular song or musical-comedy number


In [94]:
words=[w for s in tokenized_sentences for w in s]

In [97]:
pairs=Counter()
for i,w in enumerate(words):
    for j in range(max(0,i-2),min(len(words),i+3)):
        if i!=j:
            pairs[(w,words[i])]+=1
total=sum(pairs.values())
w_counts=Counter(words)
for (w1,w2),c in list(pairs.items()):
    pmi=np.log2((c/total)/((w_counts[w1]/len(words))*(w_counts[w2]/len(words))))
    print(f"{w1},{w2}->PPMI:",pmi)

i,i->PPMI: 3.8946710469754393
do,do->PPMI: 5.327630454251546
not,not->PPMI: 5.74266795353039
like,like->PPMI: 4.157705452809233
bugs,bugs->PPMI: 5.74266795353039
<sad>,<sad>->PPMI: 5.74266795353039
natural,natural->PPMI: 5.74266795353039
language,language->PPMI: 4.74266795353039
processing,processing->PPMI: 5.74266795353039
<happy>,<happy>->PPMI: 5.74266795353039
machine,machine->PPMI: 5.74266795353039
learning,learning->PPMI: 5.74266795353039
and,and->PPMI: 5.74266795353039
models,models->PPMI: 5.74266795353039
the,the->PPMI: 3.157705452809233
boy,boy->PPMI: 4.74266795353039
saw,saw->PPMI: 5.74266795353039
with,with->PPMI: 5.74266795353039
telescope,telescope->PPMI: 5.74266795353039
food,food->PPMI: 5.74266795353039
is,is->PPMI: 5.74266795353039
delicious,delicious->PPMI: 5.74266795353039
phone,phone->PPMI: 5.74266795353039
has,has->PPMI: 5.74266795353039
a,a->PPMI: 4.74266795353039
great,great->PPMI: 5.74266795353039
screen,screen->PPMI: 5.74266795353039
mr,mr->PPMI: 5.74266795353039

In [99]:
w2v=Word2Vec(sentences=tokenized_sentences,vector_size=50,window=2,min_count=1,sg=1,seed=42,workers=1,epochs=50)
w2v.wv.most_similar("learning",topn=3)

[('and', 0.33537665009498596),
 ('liked', 0.3066946864128113),
 ('<sad>', 0.22451967000961304)]

In [ ]:
doc=nlp(sentences[-1])
print("Named Entities:", [(e.text, e.label_) for e in doc.ents])
print("Noun Chunks   :", [c.text for c in doc.noun_chunks])
print("Pronouns      :", [t.text for t in doc if t.pos_ == "PRON"])
male_words = {"mr.", "mr", "rahul", "john", "he", "him", "his", "boy"}
female_words = {"mrs.", "mrs", "ms.", "priya", "sara", "she", "her", "girl"}
plural_words = {"they", "them", "their"}
